# Unsupervised Confidence Score: Gaussian Mixture Model

**Goal:** Combine 7 independent signals into a single confidence score via 2-component GMM.

**Data:** 1,984 annotated spectra (named compounds only). No ground truth labels.

**Approach:** Fit GMM unsupervised. The data should separate into "confident" (features agree) and "uncertain" (features disagree) populations. Posterior P(confident | features) = confidence score.

**Features (93.9%–100% coverage):**
1. identity — MS2 match quality
2. entropy — spectrum informativeness
3. sim_gap — uniqueness (InChIKey14-deduplicated)
4. delta_rt_pred — RT agreement (93.9%)
5. n_adducts — multi-adduct corroboration
6. delta_ppm — **mass accuracy (99.7%)** ← strongest Orbitrap signal
7. identity/fuzzy ratio — ISF signal detector

**Validation:** Oliver's 124 expert comments (post-hoc sanity check, not training).

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from scipy.stats import multivariate_normal
import seaborn as sns

warnings.filterwarnings('ignore')

ROOT = '/Users/ellayoung/Desktop/metabolo_confi_score'
OUT = f'{ROOT}/results/current'
HITS_FILE = f'{ROOT}/data/library_hits/orbitrap_hilic_neg_masswiki_hits.csv'

# Load Orbitrap spreadsheet
print('Loading Orbitrap HILIC negESI spreadsheet...')
raw = pd.read_excel(
    f'{ROOT}/data/masswiki_Orbitrap HILIC negESI_2026-03-19.xlsx',
    sheet_name='masswiki_result_2026-03-19',
    header=4
)

df = raw.rename(columns={
    'identity_score': 'identity',
    'fuzzy_score': 'fuzzy',
    'identity_score_reference_library_': 'ref_sim_1st',
    'reference_library_search-identity_score_2nd': 'ref_sim_2nd',
    'DRTpred': 'delta_rt_pred',
    'Unnamed: 36': 'oliver_comment',
    'oliver\'s ad hoc probability': 'oliver_prob',
})

# Numeric conversion
for col in ['identity', 'fuzzy', 'ref_sim_1st', 'ref_sim_2nd', 'delta_rt_pred', 'entropy', 
            'oliver_prob', 'precursor_mz']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove ISTDs
is_istd = df['name'].astype(str).str.startswith('1_')
df = df[~is_istd].copy()

# Filter to ANNOTATED spectra only
annotated = df[
    (df['name'].notna()) & 
    (df['name'] != 'UNKNOWN') & 
    (df['name'].astype(str).str.strip() != '')
].copy()

print(f'Total spectra: {len(df)}')
print(f'Annotated spectra: {len(annotated)}')

# Load MassWiki hits
hits = pd.read_csv(HITS_FILE, low_memory=False)
hits['entropy_similarity'] = pd.to_numeric(hits['entropy_similarity'], errors='coerce')
hits['lib_precursor_mz'] = pd.to_numeric(hits['lib_precursor_mz'], errors='coerce')
print(f'MassWiki hits: {len(hits)} rows')

# Compute InChIKey14-deduplicated sim_gap
from rdkit import Chem
from rdkit.Chem.inchi import InchiToInchiKey, MolToInchi

def smiles_to_inchikey14(smi):
    if not isinstance(smi, str) or not smi.strip():
        return None
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None
        ik = InchiToInchiKey(MolToInchi(mol))
        return ik[:14] if ik else None
    except Exception:
        return None

ref_hits = hits[hits['hit_source'] == 'reference'].copy()
unique_smiles = ref_hits['smiles'].dropna().unique()
smi_to_ik14 = {smi: smiles_to_inchikey14(smi) for smi in unique_smiles}
ref_hits['ik14'] = ref_hits['smiles'].map(smi_to_ik14)
ref_hits['dedup_key'] = ref_hits['ik14'].fillna(ref_hits['lib_name'])

best_per_compound = (ref_hits
    .sort_values('entropy_similarity', ascending=False)
    .groupby(['wiki_id', 'dedup_key'])
    .first()
    .reset_index()
)

ranked = (best_per_compound
    .sort_values(['wiki_id', 'entropy_similarity'], ascending=[True, False])
    .groupby('wiki_id')
)
top1 = ranked.nth(0).set_index('wiki_id')[['entropy_similarity', 'lib_precursor_mz']].rename(columns={'entropy_similarity': 'sim_1st'})
top2 = ranked.nth(1).set_index('wiki_id')[['entropy_similarity']].rename(columns={'entropy_similarity': 'sim_2nd'})

sim_gap_df = top1.join(top2)
sim_gap_df['sim_gap'] = (sim_gap_df['sim_1st'] - sim_gap_df['sim_2nd']).fillna(sim_gap_df['sim_1st'])

# Merge with annotated (keep lib_precursor_mz)
annotated = annotated.merge(
    sim_gap_df[['sim_gap', 'lib_precursor_mz']], 
    left_on='wiki_id', right_index=True, how='left'
)

# Compute n_adducts
adduct_counts = annotated.groupby('name')['adduct'].nunique().rename('n_adducts')
annotated = annotated.merge(adduct_counts, on='name', how='left')

print(f'\nReady: {len(annotated)} annotated spectra with sim_gap and lib_precursor_mz')


## Compute new features

In [ ]:
# Δppm: mass accuracy
annotated['delta_ppm'] = (
    (annotated['precursor_mz'] - annotated['lib_precursor_mz']).abs() / 
    annotated['lib_precursor_mz'] * 1e6
)

# Identity/fuzzy ratio
annotated['id_fuzzy_ratio'] = annotated['identity'] / (annotated['fuzzy'] + 1e-6)

print('\n=== Feature Coverage ===')
for col in ['identity', 'entropy', 'sim_gap', 'delta_rt_pred', 'n_adducts', 'delta_ppm', 'id_fuzzy_ratio']:
    n = annotated[col].notna().sum()
    pct = n / len(annotated) * 100
    print(f'{col:<20s}: {n:>4} / {len(annotated)} ({pct:>5.1f}%)')


## Feature Exploration

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Raw feature distributions', fontsize=13)

for ax, col in zip(axes.flat, ['identity', 'fuzzy', 'entropy', 'sim_gap', 'delta_rt_pred', 'n_adducts', 'delta_ppm', 'id_fuzzy_ratio']):
    data = annotated[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.7)
    ax.set_title(f'{col} (n={len(data)}, μ={data.mean():.2f})')
    ax.set_xlabel(col)
    
plt.tight_layout()
plt.savefig(f'{ROOT}/figures/gmm_features_raw.png', dpi=150, bbox_inches='tight')
plt.show()


## Feature Transform & Standardization

In [ ]:
# Transform features
eps = 1e-3

annotated['logit_identity'] = np.log(
    np.clip(annotated['identity'], eps, 1-eps) / (1 - np.clip(annotated['identity'], eps, 1-eps))
)
annotated['log_sim_gap'] = np.log1p(annotated['sim_gap'])
annotated['log_abs_drt'] = np.log1p(annotated['delta_rt_pred'].abs())

# GMM features
feature_cols = ['logit_identity', 'entropy', 'log_sim_gap', 'log_abs_drt', 'n_adducts', 'delta_ppm', 'id_fuzzy_ratio']

# Complete cases
complete_mask = annotated[feature_cols].notna().all(axis=1)
n_complete = complete_mask.sum()
print(f'Complete cases: {n_complete} / {len(annotated)} ({n_complete/len(annotated)*100:.1f}%)')

# Standardize
scaler = StandardScaler()
X_complete = annotated.loc[complete_mask, feature_cols].values
scaler.fit(X_complete)
X_scaled = scaler.transform(annotated[feature_cols].values)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=annotated.index)

print('\nTransformed features (complete cases):')
for col in feature_cols:
    data = X_scaled_df.loc[complete_mask, col]
    print(f'  {col:<20s}: μ={data.mean():>7.3f}, σ={data.std():>6.3f}')


## Model Selection: BIC for k=1,2,3,4

In [ ]:
# BIC comparison
print('Fitting GMM for BIC comparison...')
bics = {}
for k in [1, 2, 3, 4]:
    for cov_type in ['full', 'diag']:
        gmm = GaussianMixture(n_components=k, covariance_type=cov_type, 
                               n_init=10, random_state=42)
        gmm.fit(X_complete)
        bic = gmm.bic(X_complete)
        bics[(k, cov_type)] = bic
        print(f'  k={k}, cov={cov_type:<4s}: BIC={bic:>10.1f}')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# BIC vs k (full)
full_bics = [bics[(k, 'full')] for k in [1, 2, 3, 4]]
ax1.plot([1, 2, 3, 4], full_bics, 'o-', lw=2, markersize=8, color='steelblue')
ax1.set_xlabel('Number of components (k)')
ax1.set_ylabel('BIC')
ax1.set_title('BIC vs k (full covariance)')
ax1.grid(alpha=0.3)

# BIC vs k (diag)
diag_bics = [bics[(k, 'diag')] for k in [1, 2, 3, 4]]
ax2.plot([1, 2, 3, 4], diag_bics, 'o-', lw=2, markersize=8, color='coral')
ax2.set_xlabel('Number of components (k)')
ax2.set_ylabel('BIC')
ax2.set_title('BIC vs k (diagonal covariance)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{ROOT}/figures/gmm_bic_selection.png', dpi=150, bbox_inches='tight')
plt.show()

# Best model
best_k = min(bics, key=bics.get)[0]
best_cov = min(bics, key=bics.get)[1]
print(f'\nBest model: k={best_k}, covariance={best_cov} (BIC={bics[(best_k, best_cov)]:.1f})')


## Fit Final GMM (k=2, full covariance)

In [ ]:
# Fit k=2 GMM with full covariance
gmm = GaussianMixture(n_components=2, covariance_type='full', n_init=20, random_state=42)
gmm.fit(X_complete)

print('GMM fitted on', n_complete, 'complete cases')
print(f'\nComponent mixing weights: {gmm.weights_}')

# Identify which component is "confident" (higher mean logit_identity)
id_idx = feature_cols.index('logit_identity')
means_per_component = gmm.means_[:, id_idx]
confident_component = np.argmax(means_per_component)
uncertain_component = 1 - confident_component

print(f'\nComponent {confident_component}: CONFIDENT (higher mean logit_identity = {means_per_component[confident_component]:.2f})')
print(f'Component {uncertain_component}: UNCERTAIN (lower mean logit_identity = {means_per_component[uncertain_component]:.2f})')

# Show back-transformed component means
print(f'\n=== Component Means (back-transformed to original scale) ===')
for comp in [confident_component, uncertain_component]:
    label = 'CONFIDENT' if comp == confident_component else 'UNCERTAIN'
    print(f'\nComponent {comp} ({label}):')
    
    for j, col in enumerate(feature_cols):
        mean_val = gmm.means_[comp, j]
        
        # Back-transform
        if col == 'logit_identity':
            # logit^{-1}(x) = 1 / (1 + exp(-x))
            orig = 1.0 / (1.0 + np.exp(-mean_val))
            print(f'  {col:<20s}: {orig:.3f} (logit={mean_val:.2f})')
        elif col == 'log_sim_gap':
            orig = np.expm1(mean_val)
            print(f'  {col:<20s}: {orig:.3f} (log1p={mean_val:.2f})')
        elif col == 'log_abs_drt':
            orig = np.expm1(mean_val)
            print(f'  {col:<20s}: {orig:.1f}s (log1p={mean_val:.2f})')
        else:
            print(f'  {col:<20s}: {mean_val:.3f}')


## Score all spectra via GMM posterior

In [ ]:
# Score all 1,984 annotated spectra
# Complete cases: use gmm.predict_proba directly
# Missing ΔRT cases: marginalize via 3-feature Gaussian

def gmm_posterior_marginal(gmm, x_available, available_indices, target_component):
    '''Compute P(target_component | available features) via Gaussian marginalization.'''
    log_probs = []
    for k in range(gmm.n_components):
        mu_k = gmm.means_[k][available_indices]
        cov_k = gmm.covariances_[k][np.ix_(available_indices, available_indices)]
        lp = multivariate_normal.logpdf(x_available, mean=mu_k, cov=cov_k)
        log_probs.append(lp + np.log(gmm.weights_[k]))
    
    # Softmax
    max_lp = max(log_probs)
    probs = np.array([np.exp(lp - max_lp) for lp in log_probs])
    return probs[target_component] / probs.sum()

# Initialize
annotated['p_correct'] = np.nan
annotated['gmm_component'] = np.nan

# Complete cases
proba = gmm.predict_proba(X_complete)
annotated.loc[complete_mask, 'p_correct'] = proba[:, confident_component]
annotated.loc[complete_mask, 'gmm_component'] = gmm.predict(X_complete)

# Missing cases: marginalize
for idx in annotated.index[~complete_mask]:
    x_row = X_scaled_df.loc[idx].values
    available_mask = ~np.isnan(x_row)
    available_indices = np.where(available_mask)[0]
    
    if len(available_indices) >= 2:  # Need at least 2 features
        x_available = x_row[available_mask]
        try:
            p_conf = gmm_posterior_marginal(gmm, x_available, available_indices, confident_component)
            annotated.loc[idx, 'p_correct'] = p_conf
        except Exception:
            annotated.loc[idx, 'p_correct'] = np.nan

print(f'Scored: {annotated["p_correct"].notna().sum()} / {len(annotated)} spectra')
print(f'\np_correct distribution:')
p_vals = annotated['p_correct'].dropna()
print(f'  mean: {p_vals.mean():.3f}, median: {p_vals.median():.3f}')
print(f'  min: {p_vals.min():.3f}, max: {p_vals.max():.3f}')
print(f'  std: {p_vals.std():.3f}')


## Results: Confidence Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(annotated['p_correct'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.7)
ax.axvline(0.2, color='red', ls='--', lw=1.5, label='Very Low (0.2)')
ax.axvline(0.5, color='orange', ls='--', lw=1.5, label='Low (0.5)')
ax.axvline(0.8, color='green', ls='--', lw=1.5, label='High (0.8)')
ax.set_xlabel('p_correct (confidence score)')
ax.set_ylabel('Count')
ax.set_title(f'Confidence distribution (n={annotated["p_correct"].notna().sum()})')
ax.legend(fontsize=9)

# Tier breakdown
ax = axes[1]
tiers = ['Very Low\n(<0.2)', 'Low\n(0.2-0.5)', 'Medium\n(0.5-0.8)', 'High\n(>0.8)']
tier_counts = [
    (annotated['p_correct'] < 0.2).sum(),
    ((annotated['p_correct'] >= 0.2) & (annotated['p_correct'] < 0.5)).sum(),
    ((annotated['p_correct'] >= 0.5) & (annotated['p_correct'] < 0.8)).sum(),
    (annotated['p_correct'] >= 0.8).sum(),
]
colors = ['crimson', 'orange', 'gold', 'forestgreen']
ax.bar(tiers, tier_counts, color=colors, edgecolor='white', alpha=0.7)
ax.set_ylabel('Count')
ax.set_title('Confidence tiers')
for i, (tier, count) in enumerate(zip(tiers, tier_counts)):
    ax.text(i, count + 20, str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{ROOT}/figures/gmm_confidence_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Confidence Tier Summary ===')
for (low, high), label in [((0, 0.2), 'Very Low'), ((0.2, 0.5), 'Low'), ((0.5, 0.8), 'Medium'), ((0.8, 1.0), 'High')]:
    mask = (annotated['p_correct'] >= low) & (annotated['p_correct'] < high)
    n = mask.sum()
    print(f'{label:<12s} ({low:.1f}–{high:.1f}): {n:>4} spectra ({n/len(annotated)*100:>5.1f}%)')


## Validation: Oliver's 124 Comments

In [ ]:
# Categorize Oliver's comments
isf_keywords = ['isf', 'in-source', 'insource', 'in source', 'fragment']
msms_keywords = ['msms ok', 'msms', 'ok']

annotated['comment_category'] = 'Other'
annotated.loc[annotated['oliver_comment'].fillna('').str.lower().apply(lambda x: any(k in x for k in isf_keywords)), 'comment_category'] = 'ISF'
annotated.loc[annotated['oliver_comment'].fillna('').str.lower().apply(lambda x: any(k in x for k in msms_keywords)), 'comment_category'] = 'MSMS OK'
annotated.loc[annotated['oliver_comment'].fillna('').str.lower().str.contains('no alternative'), 'comment_category'] = 'No Alternative'

# Summary
print('Oliver\'s commented cases:')
for cat in annotated['comment_category'].unique():
    if pd.notna(cat):
        sub = annotated[annotated['comment_category'] == cat]
        p_vals = sub['p_correct'].dropna()
        print(f'  {cat:<20s}: {len(sub):>3} cases, p_correct μ={p_vals.mean():.3f}')

# Boxplot
fig, ax = plt.subplots(figsize=(10, 6))
commented = annotated[annotated['oliver_comment'].notna()].copy()
sns.boxplot(data=commented, x='comment_category', y='p_correct', ax=ax, palette='Set2')
ax.set_ylabel('p_correct')
ax.set_xlabel('Comment category')
ax.set_title('Confidence scores by Oliver\'s comment category')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{ROOT}/figures/gmm_validation_oliver.png', dpi=150, bbox_inches='tight')
plt.show()


## Export Results

In [ ]:
# Define confidence tiers
def tier_label(p):
    if pd.isna(p):
        return 'Unknown'
    elif p < 0.2:
        return 'Very Low'
    elif p < 0.5:
        return 'Low'
    elif p < 0.8:
        return 'Medium'
    else:
        return 'High'

annotated['confidence_tier'] = annotated['p_correct'].apply(tier_label)

# Export columns
export_cols = [
    'wiki_id', 'name', 'adduct', 'precursor_mz', 'rt',
    'identity', 'fuzzy', 'entropy', 'sim_gap', 'delta_rt_pred', 'n_adducts', 'delta_ppm', 'id_fuzzy_ratio',
    'p_correct', 'confidence_tier', 'gmm_component',
    'oliver_prob', 'oliver_comment'
]

export_df = annotated[[c for c in export_cols if c in annotated.columns]].copy()
export_df = export_df.sort_values('p_correct')

export_path = f'{OUT}/confidence_scores_gmm.csv'
export_df.to_csv(export_path, index=False)
print(f'Saved {len(export_df)} scores → {export_path}')

print(f'\n=== Export Summary ===')
print(f'Columns: {len(export_df.columns)}')
print(f'Rows: {len(export_df)}')
print(f'p_correct range: [{export_df["p_correct"].min():.3f}, {export_df["p_correct"].max():.3f}]')
print(f'\nTier distribution in export:')
print(export_df['confidence_tier'].value_counts().sort_index())
